In [ ]:
import nbformat
import plotly.graph_objects as go
import plotly.subplots as sp
import WTdelineator as wav
import wfdb
import numpy as np
from scipy import signal
import neurokit2 as nk
import numpy as np

## Cache data to avoid re-downloading
import os
import pickle



In [19]:
import wfdb
import matplotlib.pyplot as plt

# Load the signal from data/staff_III/001c
record = wfdb.rdrecord('data/staff_III/data/test')
staff_signal = record.p_signal[:2000].T # type: ignore
staff_sampling_rate = record.fs
staff_channel_names = record.sig_name

print(f"Signal shape: {staff_signal.shape}")
print(f"Sampling rate: {staff_sampling_rate} Hz")
print(f"Number of channels: {staff_signal.shape[1]}")
print(f"Channel names: {staff_channel_names}")


Signal shape: (9, 2000)
Sampling rate: 1000 Hz
Number of channels: 2000
Channel names: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'I', 'II', 'III']


In [12]:
record.n_sig

9

In [ ]:
dbase = "staffiii/data"
rec = "052c"
sNum = 1

cache_dir = "data/staff_III"
os.makedirs(cache_dir, exist_ok=True)

cache_file_s = os.path.join(cache_dir, f"{rec}_signal.pkl")
cache_file_att = os.path.join(cache_dir, f"{rec}_attributes.pkl")

if os.path.exists(cache_file_s) and os.path.exists(cache_file_att):
    # Load from cache
    print(f"Loading cached data from {cache_dir}...")
    with open(cache_file_s, 'rb') as f:
        s = pickle.load(f)
    with open(cache_file_att, 'rb') as f:
        att = pickle.load(f)
    print("Data loaded from cache.\n")
else:
    # Download and cache
    print("Downloading data from PhysioNet...")
    s, att = wfdb.rdsamp(rec, pn_dir=dbase)
    print("Caching data...")
    with open(cache_file_s, 'wb') as f:
        pickle.dump(s, f)
    with open(cache_file_att, 'wb') as f:
        pickle.dump(att, f)
    print(f"Data cached to {cache_dir}")

annot = wfdb.rdann(rec, "event", pn_dir=dbase)
sName = att["sig_name"]

# Ranges to analyse signal
beg = int(np.floor(2**16))
end = int(np.floor(2 * 2**16))

print("Signal array shape:", s.shape)
print("Number of leads:", s.shape[1])
print("Total samples:", s.shape[0])
print("\nSampling frequency:", att["fs"], "Hz")
print("Duration:", s.shape[0] / att["fs"], "seconds")
print("\nSignal names:", sName)

Loading cached data from data/staff_III...
Data loaded from cache.

Signal array shape: (500472, 9)
Number of leads: 9
Total samples: 500472

Sampling frequency: 1000 Hz
Duration: 500.472 seconds

Signal names: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'I', 'II', 'III']


In [6]:
# # Plot all ECG leads using Plotly
# fs = att["fs"]
# t_full = np.arange(s.shape[0]) / fs

# n_leads = s.shape[1]
# fig = sp.make_subplots(rows=n_leads, cols=1, shared_xaxes=True, 
#                         subplot_titles=sName, vertical_spacing=0.05)

# for i, name in enumerate(sName):
#     fig.add_trace(
#         go.Scatter(x=t_full, y=s[:, i], name=name, mode='lines', line=dict(width=0.5)),
#         row=i+1, col=1
#     )
#     fig.update_yaxes(title_text=name, row=i+1, col=1)

# fig.update_xaxes(title_text="Time (s)", row=n_leads, col=1)
# fig.update_layout(title="All ECG Leads - Full Recording", height=300*n_leads, hovermode='x unified')
# fig.show()

In [ ]:
fs = att["fs"]
sig = s[beg:end, sNum]
N = sig.shape[0]
t = np.arange(0, N / fs, 1 / fs)

# Wavelet Transform delineation
Pwav, QRS, Twav = wav.signalDelineation(sig, fs)

# Calculate biomarkers
QRSd = QRS[:, -1] - QRS[:, 0]

Tind = np.nonzero(Twav[:, 0])
QT = Twav[Tind, -1] - QRS[Tind, 0]
Td = Twav[Tind, -1] - Twav[Tind, 0]

Pind = np.nonzero(Pwav[:, 0])
Pd = Pwav[Pind, -1] - Pwav[Pind, 0]

fig = go.Figure()

fig.add_trace(go.Scatter(x=t, y=sig, mode='lines', name=sName[sNum], line=dict(color='steelblue', width=1)))

marker_traces = [
    (Pwav[:, 0], 'Pon',   'circle', 'orange'),
    (Pwav[:, 1], 'P1',    'circle', 'black'),
    (Pwav[:, 2], 'P2',    'circle', 'magenta'),
    (Pwav[:, 3], 'Pend',  'circle', 'green'),
    (QRS[:, 0], 'QRSon',  'star',        'red'),
    (QRS[:, 1], 'Q',      'star',        'gold'),
    (QRS[:, 2], 'R',      'star',        'black'),
    (QRS[:, 3], 'S',      'star',        'magenta'),
    (QRS[:, 4], 'QRSend', 'star',        'green'),
    (Twav[:, 0], 'Ton',   'triangle-up', 'red'),
    (Twav[:, 1], 'T1',    'triangle-up', 'black'),
    (Twav[:, 2], 'T2',    'triangle-up', 'magenta'),
    (Twav[:, 3], 'Tend',  'triangle-up', 'green'),
]

for idx, label, symbol, color in marker_traces:
    fig.add_trace(go.Scatter(
        x=t[idx], y=sig[idx],
        mode='markers',
        name=label,
        marker=dict(symbol=symbol, color=color, size=10)
    ))

fig.update_layout(
    title="Delineator output",
    xaxis_title="Time (s)",
    yaxis_title="ECG (mV)",
    hovermode='x unified'
)
fig.show()


In [8]:
# Extract ecg_signal and rpeaks from delineation results
ecg_signal = sig  # The extracted ECG signal
rpeaks = QRS[QRS[:, 2] > 0, 2].astype(int)  # R-peak indices (column 2 of QRS matrix)

print(f"ECG Signal shape: {ecg_signal.shape}")
print(f"Number of R-peaks detected: {len(rpeaks)}")
print(f"R-peak indices (first 10): {rpeaks[:10]}")


ECG Signal shape: (65536,)
Number of R-peaks detected: 67
R-peak indices (first 10): [ 742 1643 2548 3466 4403 5359 6316 7267 8210 9132]


In [ ]:
# Assume you have your ECG signal and delineator results
# rpeaks, t_onsets, etc. from wavelet tool
signals, waves = nk.ecg_delineate(ecg_signal, rpeaks, sampling_rate=fs, method="dwt")

# Custom ST measurement using your boundaries
def measure_st_elevation(ecg, j_point, sampling_rate, offset_ms=80):
    offset_samples = int(offset_ms * sampling_rate / 1000)
    st_point = j_point + offset_samples
    baseline = np.mean(ecg[some_PR_segment])  # e.g., from P-end to QRS-onset
    return ecg[st_point] - baseline  # in same units as signal (usually mV or µV)

# Thresholds (e.g., >1mm in limb leads, >2mm in precordial per guidelines)

In [15]:
import glob
import pandas as pd
import os

# Get all .hea files in data/staff_III/data
hea_files = glob.glob('data/staff_III/data/*.hea')

# Extract record names (without extension and path)
record_names = [os.path.basename(f)[:-4] for f in sorted(hea_files)]

# Load each record and extract n_sig
n_sig_values = []
for rec_name in record_names:
    try:
        rec = wfdb.rdrecord(f'data/staff_III/data/{rec_name}')
        n_sig_values.append(rec.n_sig)
    except Exception as e:
        print(f"Error loading {rec_name}: {e}")
        n_sig_values.append(None)

# Create Series
n_sig_series = pd.Series(n_sig_values, index=record_names, name='n_sig')

print(f"Total records: {len(n_sig_series)}")
print(f"\nn_sig distribution:")
print(n_sig_series.value_counts().sort_index())
print(f"\nSeries:")
print(n_sig_series)

# Save to pickle
n_sig_series.to_pickle('data/staff_III/n_sig_series.pkl')
print("\nSaved to data/staff_III/n_sig_series.pkl")

Error loading 089d: Samples were not loaded correctly
Total records: 520

n_sig distribution:
n_sig
9.0    519
Name: count, dtype: int64

Series:
001a    9.0
001b    9.0
001c    9.0
001d    9.0
002a    9.0
       ... 
108a    9.0
108b    9.0
108c    9.0
108d    9.0
108e    9.0
Name: n_sig, Length: 520, dtype: float64

Saved to data/staff_III/n_sig_series.pkl


In [17]:
n_sig_series.value_counts()

n_sig
9.0    519
Name: count, dtype: int64